# 8. Chi-Square Tests

**Statistical Foundations for Data Science — Notebook 8 of 8**

t-tests need means, and means need numbers. But a great deal of real data is **categorical**:
survived / died, clicked / bounced, north / south / east / west, satisfied / neutral /
unhappy. You cannot average "north". All you have are **counts**.

The chi-square family of tests works entirely with counts, by comparing what you
**observed** against what you would **expect** if the null hypothesis were true.

### What you will learn

1. The chi-square statistic and where its distribution comes from
2. **Goodness-of-fit test** — does one categorical variable match an expected distribution?
3. **Test of independence** — are two categorical variables related?
4. **Test of homogeneity** — do several populations have the same distribution?
5. Building and reading **contingency tables**
6. **Assumptions**, expected-frequency rules, and **Yates' correction**
7. **Fisher's exact test** for small samples
8. **Effect size**: Cramér's V and the phi coefficient
9. Residual analysis — *which* cells drive the result
10. Applications: A/B testing, feature selection, model calibration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

rng = np.random.default_rng(seed=808)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)
pd.set_option("display.width", 120)

---
## 8.1 The chi-square statistic

Every test in this notebook uses the same statistic:

$$\chi^2 = \sum_{\text{all cells}} \frac{(O_i - E_i)^2}{E_i}$$

where $O_i$ is the **observed** count in a cell and $E_i$ is the count **expected under
$H_0$**.

Read the formula as three ideas:

1. $(O - E)$ — how far off are we?
2. Square it — so overshoots and undershoots both count as evidence
3. Divide by $E$ — a discrepancy of 10 is huge when you expected 5, trivial when you
   expected 5,000

Large $\chi^2$ means the observed pattern is far from what $H_0$ predicts. It is always a
**right-tailed** test: only large values are evidence against $H_0$.

### Where the distribution comes from

Recall from Notebook 3 that a chi-square distribution with $k$ degrees of freedom is the
sum of $k$ squared standard Normals. Each standardised cell discrepancy
$(O-E)/\sqrt{E}$ is approximately standard Normal for large counts, so their squares sum to
approximately $\chi^2$. That is the entire justification — and it is why the test needs
**adequately large expected counts**.

In [ ]:
# A first example computed by hand: is a die fair?
observed = np.array([43, 52, 68, 61, 39, 57])          # 320 rolls
n_rolls = observed.sum()
expected = np.full(6, n_rolls / 6)

contrib = (observed - expected) ** 2 / expected
chi2 = contrib.sum()
df = len(observed) - 1
p = stats.chi2(df).sf(chi2)

table = pd.DataFrame({
    "face": range(1, 7),
    "observed": observed,
    "expected": expected,
    "O - E": observed - expected,
    "(O-E)^2/E": contrib.round(4),
})
print(table.to_string(index=False))
print(f"\nchi2 = {chi2:.4f}   df = {df}   p = {p:.5f}")
print(f"Critical value at 0.05 = {stats.chi2(df).ppf(0.95):.4f}")
print(f"Decision: {'REJECT H0 -- the die looks biased' if p < 0.05 else 'fail to reject -- consistent with a fair die'}")
print()
print("scipy in one line:")
print(f"  {stats.chisquare(observed)}")

In [ ]:
# Visualise the null distribution and the observed statistic
xs = np.linspace(0.01, 25, 600)
pdf = stats.chi2(df).pdf(xs)

plt.plot(xs, pdf, color="black", lw=1.8, label=f"chi2 with df={df}")
tail = xs >= chi2
plt.fill_between(xs[tail], pdf[tail], color="crimson", alpha=0.6, label=f"p = {p:.4f}")
plt.axvline(chi2, color="crimson", lw=2, ls="--", label=f"observed chi2 = {chi2:.2f}")
plt.axvline(stats.chi2(df).ppf(0.95), color="steelblue", lw=1.4, ls=":",
            label=f"critical value = {stats.chi2(df).ppf(0.95):.2f}")
plt.xlabel("chi-square statistic"); plt.ylabel("density")
plt.title("Chi-square is always a right-tailed test")
plt.legend(fontsize=8)
plt.show()

---
## 8.2 Goodness-of-fit test

**Question:** does the distribution of *one* categorical variable match a hypothesised
distribution?

$$H_0: p_1 = \pi_1,\ p_2 = \pi_2,\ \dots,\ p_k = \pi_k \qquad
H_1: \text{at least one } p_i \ne \pi_i$$

$$E_i = n \pi_i, \qquad df = k - 1$$

Why $k-1$? The proportions must sum to 1, so once you know $k-1$ of them the last is fixed.

The hypothesised proportions can be **equal** (a fair die) or **unequal** (a known
population breakdown, a business forecast, a theoretical genetic ratio).

In [ ]:
# Unequal expected proportions: does our customer base match the national age mix?
national = pd.DataFrame({
    "age_band": ["18-24", "25-34", "35-49", "50-64", "65+"],
    "national_share": [0.14, 0.22, 0.28, 0.24, 0.12],
    "observed": [312, 486, 402, 218, 82],
})
n_total = national["observed"].sum()
national["expected"] = national["national_share"] * n_total
national["O - E"] = national["observed"] - national["expected"]
national["contribution"] = (national["O - E"] ** 2) / national["expected"]

print(national.round(2).to_string(index=False))
print(f"\nTotal customers: {n_total}")

res = stats.chisquare(f_obs=national["observed"], f_exp=national["expected"])
print(f"chi2 = {res.statistic:.4f}, df = {len(national)-1}, p = {res.pvalue:.3e}")
print(f"\nDecision: {'REJECT H0' if res.pvalue < 0.05 else 'fail to reject'} -- our customers")
print("do NOT mirror the national age distribution.")
print(f"\nBiggest contributors: {national.nlargest(2, 'contribution')['age_band'].tolist()}")
print("We over-index on 25-34 and under-index on 50-64.")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))

x = np.arange(len(national))
w = 0.38
ax[0].bar(x - w/2, national["observed"], w, label="observed", color="steelblue")
ax[0].bar(x + w/2, national["expected"], w, label="expected", color="darkorange")
ax[0].set_xticks(x); ax[0].set_xticklabels(national["age_band"])
ax[0].set_ylabel("customers"); ax[0].set_title("Observed vs expected counts")
ax[0].legend(fontsize=8)

colours = ["crimson" if v > 0 else "steelblue" for v in national["O - E"]]
ax[1].bar(x, national["O - E"], color=colours)
ax[1].axhline(0, color="black", lw=1)
ax[1].set_xticks(x); ax[1].set_xticklabels(national["age_band"])
ax[1].set_title("Where the discrepancy lives (O - E)")
plt.tight_layout(); plt.show()

In [ ]:
# Goodness-of-fit is not only for demographics: check whether counts follow a Poisson law.
support_tickets = np.array([12, 31, 45, 39, 28, 15, 8, 2])     # observed days with 0,1,...,7+ tickets
k_values = np.arange(len(support_tickets))
n_days = support_tickets.sum()

lambda_hat = (k_values * support_tickets).sum() / n_days        # MLE of lambda
probs = stats.poisson(lambda_hat).pmf(k_values)
probs[-1] += stats.poisson(lambda_hat).sf(k_values[-1])        # lump the tail into the last bin
exp_counts = probs * n_days

fit = pd.DataFrame({"tickets": k_values, "observed_days": support_tickets,
                    "expected_days": exp_counts.round(2)})
print(f"Estimated lambda = {lambda_hat:.3f}")
print(fit.to_string(index=False))

# df = bins - 1 - (parameters estimated from the data) = 8 - 1 - 1
chi2_fit = ((support_tickets - exp_counts) ** 2 / exp_counts).sum()
df_fit = len(k_values) - 1 - 1
print(f"\nchi2 = {chi2_fit:.4f}, df = {df_fit}, p = {stats.chi2(df_fit).sf(chi2_fit):.4f}")
print("Note the extra degree of freedom spent estimating lambda from the same data.")

---
## 8.3 Test of independence

**Question:** are two categorical variables associated?

$$H_0: \text{the two variables are independent} \qquad
H_1: \text{they are associated}$$

Under independence, $P(A \cap B) = P(A)P(B)$ (straight from Notebook 1). Applied to a
contingency table, that gives the expected count for each cell:

$$E_{ij} = \frac{(\text{row } i \text{ total}) \times (\text{column } j \text{ total})}{\text{grand total}}$$

$$df = (r - 1)(c - 1)$$

**Important:** the test tells you *whether* the variables are associated, not *which causes
which*, and not *how strongly*. For strength, use Cramér's V (section 8.6).

In [ ]:
# Does device type relate to whether a user subscribes?
observed_tbl = pd.DataFrame(
    [[145,  355],       # mobile:  subscribed, not subscribed
     [190,  260],       # desktop
     [ 65,  185]],      # tablet
    index=["mobile", "desktop", "tablet"],
    columns=["subscribed", "not_subscribed"],
)
observed_tbl.index.name = "device"

with_margins = observed_tbl.copy()
with_margins["row_total"] = with_margins.sum(axis=1)
with_margins.loc["col_total"] = with_margins.sum()
print("Contingency table with margins:")
print(with_margins)

In [ ]:
# Compute the expected counts by hand
row_tot = observed_tbl.sum(axis=1).to_numpy()[:, None]
col_tot = observed_tbl.sum(axis=0).to_numpy()[None, :]
grand = observed_tbl.to_numpy().sum()
expected_tbl = pd.DataFrame(row_tot * col_tot / grand,
                            index=observed_tbl.index, columns=observed_tbl.columns)

print("Expected counts under independence (row total x col total / grand total):")
print(expected_tbl.round(2))

chi2_manual = (((observed_tbl - expected_tbl) ** 2) / expected_tbl).to_numpy().sum()
df_manual = (observed_tbl.shape[0] - 1) * (observed_tbl.shape[1] - 1)
print(f"\nchi2 = {chi2_manual:.4f},  df = ({observed_tbl.shape[0]}-1)*({observed_tbl.shape[1]}-1) = {df_manual}")
print(f"p    = {stats.chi2(df_manual).sf(chi2_manual):.3e}")

In [ ]:
# scipy does all of it, and returns the expected table too
chi2_s, p_s, dof_s, expected_s = stats.chi2_contingency(observed_tbl)
print(f"chi2 = {chi2_s:.4f}   df = {dof_s}   p = {p_s:.3e}")
print(f"Minimum expected count = {expected_s.min():.1f}  "
      f"({'assumption satisfied' if expected_s.min() >= 5 else 'TOO SMALL'})")

rates = (observed_tbl["subscribed"] / observed_tbl.sum(axis=1)).sort_values(ascending=False)
print("\nSubscription rate by device:")
for dev, r in rates.items():
    print(f"  {dev:<8} {r:.3f}   {'#' * int(r*100)}")
print(f"\nOverall rate: {observed_tbl['subscribed'].sum()/grand:.3f}")
print("Desktop users subscribe far more often; tablet users least.")

### Residual analysis: which cells drive the result?

A significant $\chi^2$ tells you *something* is going on, but not *where*. **Standardised
(Pearson) residuals** localise it:

$$r_{ij} = \frac{O_{ij} - E_{ij}}{\sqrt{E_{ij}}}$$

Better still, **adjusted residuals** correct for the row and column totals and are
approximately standard Normal, so $|r| > 2$ flags a cell worth talking about:

$$r^{adj}_{ij} = \frac{O_{ij} - E_{ij}}{\sqrt{E_{ij}\left(1 - \frac{R_i}{N}\right)\left(1 - \frac{C_j}{N}\right)}}$$

In [ ]:
O = observed_tbl.to_numpy()
E = expected_s
N = O.sum()
R = O.sum(axis=1)[:, None]
C = O.sum(axis=0)[None, :]

pearson_res = (O - E) / np.sqrt(E)
adj_res = (O - E) / np.sqrt(E * (1 - R/N) * (1 - C/N))

res_frame = pd.DataFrame(adj_res.round(2), index=observed_tbl.index, columns=observed_tbl.columns)
print("Adjusted residuals (|r| > 2 is noteworthy):")
print(res_frame)

fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
sns.heatmap(observed_tbl, annot=True, fmt="d", cmap="Blues", ax=ax[0], cbar=False)
ax[0].set_title("Observed counts")
sns.heatmap(res_frame, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            vmin=-4, vmax=4, ax=ax[1])
ax[1].set_title("Adjusted residuals")
plt.tight_layout(); plt.show()

flag = np.abs(adj_res) > 2
for i, dev in enumerate(observed_tbl.index):
    for j, col in enumerate(observed_tbl.columns):
        if flag[i, j]:
            direction = "MORE" if adj_res[i, j] > 0 else "FEWER"
            print(f"  {dev} / {col}: {direction} than expected (r = {adj_res[i,j]:+.2f})")

---
## 8.4 Test of homogeneity

The arithmetic is **identical** to the test of independence — only the study design and the
wording of the hypothesis differ.

| | Independence | Homogeneity |
|---|---|---|
| Sampling | **One** sample, two variables recorded | **Several** samples, one variable recorded |
| $H_0$ | The two variables are independent | All populations have the same distribution |
| Example | Survey 1,000 people, record device *and* subscription | Sample 300 from each of 3 cities, record satisfaction |

Because the computation is the same, `chi2_contingency` serves both. Just be careful to
describe your conclusion in the language that matches your design.

In [ ]:
# Homogeneity: we deliberately sampled 300 customers from each of three cities.
homog = pd.DataFrame(
    [[ 62, 138, 100],
     [105, 130,  65],
     [ 88, 142,  70]],
    index=["Chennai", "Bengaluru", "Hyderabad"],
    columns=["unhappy", "neutral", "satisfied"],
)
homog.index.name = "city"

chi2_h, p_h, dof_h, exp_h = stats.chi2_contingency(homog)
print(homog)
print(f"\nRow totals (by design): {homog.sum(axis=1).tolist()}")
print(f"chi2 = {chi2_h:.4f}, df = {dof_h}, p = {p_h:.5f}")
print(f"\nH0: satisfaction distribution is the same in all three cities")
print(f"Conclusion: {'REJECT -- cities differ' if p_h < 0.05 else 'no evidence of a difference'}")

props = homog.div(homog.sum(axis=1), axis=0)
props.plot(kind="bar", stacked=True, colormap="RdYlGn", figsize=(8, 4))
plt.ylabel("proportion"); plt.title("Satisfaction mix by city")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()

---
## 8.5 Assumptions and small samples

The chi-square test relies on a Normal approximation, which needs enough data in every cell.

**The rules:**

1. **Independent observations** — each subject appears in exactly one cell. *Never* run a
   chi-square test on paired data (use **McNemar's test** for that).
2. **Counts, not percentages** — feeding proportions into the formula gives nonsense.
3. **Expected counts:** all $E_{ij} \ge 5$ is the classic rule. The workable modern version:
   no expected count below 1, and at most 20% of cells below 5.
4. **Fixed total** — the sample size is not itself random.

**When the expected counts are too small:**

- Combine sparse categories into a meaningful "other"
- Use **Fisher's exact test** (exact for any sample size, originally for 2×2)
- Use a **permutation** version of the chi-square test
- Collect more data

In [ ]:
# What goes wrong with small expected counts: the p-value becomes unreliable.
small = np.array([[3, 9], [8, 4]])
chi2_p, p_plain, _, exp_small = stats.chi2_contingency(small, correction=False)
chi2_y, p_yates, _, _         = stats.chi2_contingency(small, correction=True)
odds, p_fisher                = stats.fisher_exact(small)

print("A 2x2 table with small counts:")
print(pd.DataFrame(small, index=["group A", "group B"], columns=["yes", "no"]))
print(f"\nExpected counts:\n{np.round(exp_small, 2)}")
print(f"Minimum expected count = {exp_small.min():.2f}  "
      f"({'below 5 -> approximation is shaky' if exp_small.min() < 5 else 'fine'})\n")
print(f"Chi-square, no correction : chi2 = {chi2_p:.4f}, p = {p_plain:.4f}")
print(f"Chi-square, Yates         : chi2 = {chi2_y:.4f}, p = {p_yates:.4f}")
print(f"Fisher's exact test       : odds ratio = {odds:.4f}, p = {p_fisher:.4f}  <- trust this one")
print("\nThree different answers from the same data. With small counts, use Fisher.")

### Yates' continuity correction

For 2×2 tables, `scipy` applies **Yates' correction** by default:

$$\chi^2_{\text{Yates}} = \sum \frac{(|O - E| - 0.5)^2}{E}$$

It compensates for using a continuous distribution to approximate discrete counts. It is
conservative — sometimes *too* conservative. Current practice: for 2×2 tables with small
counts prefer **Fisher's exact test**; with large counts the correction hardly matters. Be
aware of the default, because it will make your hand calculation disagree with `scipy`.

In [ ]:
print("How much does the correction matter as counts grow?")
print(f"{'scale':>7} {'min E':>8} {'p uncorrected':>15} {'p Yates':>10} {'p Fisher':>10}")
base = np.array([[3, 9], [8, 4]])
for scale in (1, 2, 5, 10, 30):
    tbl = base * scale
    _, p_u, _, e = stats.chi2_contingency(tbl, correction=False)
    _, p_y, _, _ = stats.chi2_contingency(tbl, correction=True)
    _, p_f       = stats.fisher_exact(tbl)
    print(f"{scale:>7} {e.min():>8.1f} {p_u:>15.5f} {p_y:>10.5f} {p_f:>10.5f}")
print("\nWith large counts all three agree. The disagreement is purely a small-sample issue.")

In [ ]:
# A permutation version of the chi-square test -- valid for any cell sizes.
def chi2_permutation(table, n_perm=20_000, seed=0):
    '''Permutation p-value for a contingency table, no expected-count assumption.'''
    g = np.random.default_rng(seed)
    table = np.asarray(table)
    obs_stat = stats.chi2_contingency(table, correction=False)[0]

    # Expand the table into two label arrays, then shuffle one of them
    rows, cols = [], []
    for i in range(table.shape[0]):
        for j in range(table.shape[1]):
            rows += [i] * table[i, j]
            cols += [j] * table[i, j]
    rows, cols = np.array(rows), np.array(cols)

    count = 0
    for _ in range(n_perm):
        shuffled = g.permutation(cols)
        perm_tbl = np.zeros_like(table)
        np.add.at(perm_tbl, (rows, shuffled), 1)
        if stats.chi2_contingency(perm_tbl, correction=False)[0] >= obs_stat:
            count += 1
    return obs_stat, (count + 1) / (n_perm + 1)

stat_perm, p_perm = chi2_permutation(small, n_perm=5_000)
print(f"Permutation chi-square: statistic = {stat_perm:.4f}, p = {p_perm:.4f}")
print(f"Fisher's exact        : p = {p_fisher:.4f}")
print("The permutation test agrees with Fisher, as it should.")

### Paired categorical data: McNemar's test

If the same subjects are measured twice (before/after, two raters on the same items), the
observations are **not independent** and chi-square is invalid. Use **McNemar's test**,
which looks only at the *discordant* pairs — the ones that changed.

In [ ]:
# 200 people asked whether they would recommend the product, before and after a redesign
mcnemar_tbl = np.array([[62, 41],      # before yes: after yes, after no
                        [18, 79]])     # before no : after yes, after no

print(pd.DataFrame(mcnemar_tbl,
                   index=["before: yes", "before: no"],
                   columns=["after: yes", "after: no"]))
b, c = mcnemar_tbl[0, 1], mcnemar_tbl[1, 0]
print(f"\nDiscordant pairs: {b} switched yes->no, {c} switched no->yes")

# Exact McNemar = binomial test on the discordant pairs
exact = stats.binomtest(b, b + c, p=0.5)
chi2_mc = (abs(b - c) - 1) ** 2 / (b + c)          # with continuity correction
print(f"McNemar chi2 = {chi2_mc:.4f}, p = {stats.chi2(1).sf(chi2_mc):.5f}")
print(f"Exact (binomial) p = {exact.pvalue:.5f}")
print("\nA plain chi-square test here would be WRONG -- the same 200 people appear twice.")

---
## 8.6 Effect size: Cramér's V and phi

Just like the t-test, a significant $\chi^2$ says nothing about **how strong** the
association is. With $n = 100{,}000$, a trivial association is significant.

**Cramér's V** normalises $\chi^2$ to the range $[0, 1]$:

$$V = \sqrt{\frac{\chi^2}{n \cdot \min(r-1,\ c-1)}}$$

| $V$ | Interpretation |
|---|---|
| < 0.10 | negligible |
| 0.10 – 0.20 | weak |
| 0.20 – 0.40 | moderate |
| > 0.40 | strong |

For a 2×2 table, $V$ equals the **phi coefficient** $\phi = \sqrt{\chi^2/n}$, which is also
just the Pearson correlation between the two binary variables.

In [ ]:
def cramers_v(table):
    '''Cramer's V effect size for a contingency table.'''
    table = np.asarray(table)
    chi2 = stats.chi2_contingency(table, correction=False)[0]
    n = table.sum()
    return np.sqrt(chi2 / (n * (min(table.shape) - 1)))

print(f"Device/subscription table: chi2 = {chi2_s:.2f}, p = {p_s:.2e}")
print(f"  Cramer's V = {cramers_v(observed_tbl):.4f}  -> a weak-to-moderate association")
print(f"City/satisfaction table  : Cramer's V = {cramers_v(homog):.4f}")

# 2x2: V equals phi equals the correlation of the two binary variables
two_by_two = np.array([[220, 180], [140, 260]])
phi = np.sqrt(stats.chi2_contingency(two_by_two, correction=False)[0] / two_by_two.sum())
x_bin = np.r_[np.ones(400), np.zeros(400)]
y_bin = np.r_[np.ones(220), np.zeros(180), np.ones(140), np.zeros(260)]
print(f"\n2x2 table: phi = {phi:.4f}")
print(f"           Cramer's V = {cramers_v(two_by_two):.4f}")
print(f"           Pearson r of the two binary columns = {np.corrcoef(x_bin, y_bin)[0,1]:.4f}")

In [ ]:
# The same lesson as Notebook 6: significance grows with n, effect size does not.
print("A fixed, weak association measured at different sample sizes:")
print(f"{'n':>10} {'chi2':>10} {'p-value':>12} {'Cramer V':>10}")
base_props = np.array([[0.27, 0.23], [0.23, 0.27]])      # slight association
for n in (100, 500, 2_000, 20_000, 200_000):
    tbl = (base_props * n).round().astype(int)
    c2, pv, _, _ = stats.chi2_contingency(tbl, correction=False)
    print(f"{n:>10,} {c2:>10.3f} {pv:>12.2e} {cramers_v(tbl):>10.4f}")
print("\nCramer's V is flat. Only the p-value moves. Always report both.")

---
## 8.7 Applications in data science

### (a) A/B testing with a binary outcome

When your metric is a **conversion rate** rather than a mean, chi-square (or the equivalent
two-proportion z-test) is the natural test.

In [ ]:
# A/B test: 2 variants, converted vs not
ab = pd.DataFrame([[412, 5588],     # control:  converted, not converted
                   [468, 5532]],    # variant
                  index=["control", "variant"],
                  columns=["converted", "not_converted"])

n_a, n_b = ab.sum(axis=1)
p_a, p_b = ab["converted"] / ab.sum(axis=1)

chi2_ab, p_ab, dof_ab, exp_ab = stats.chi2_contingency(ab, correction=False)

# The equivalent two-proportion z-test
p_pool = ab["converted"].sum() / ab.to_numpy().sum()
se_pool = np.sqrt(p_pool * (1 - p_pool) * (1/n_a + 1/n_b))
z = (p_b - p_a) / se_pool

print(ab)
print(f"\nControl conversion : {p_a:.4f}")
print(f"Variant conversion : {p_b:.4f}")
print(f"Absolute lift      : {p_b - p_a:+.4f}  ({(p_b/p_a - 1)*100:+.2f}% relative)")
print(f"\nChi-square : chi2 = {chi2_ab:.4f}, df = {dof_ab}, p = {p_ab:.5f}")
print(f"z-test     : z = {z:.4f}, p = {2*stats.norm.sf(abs(z)):.5f}")
print(f"Identity check: z^2 = {z**2:.4f} = chi2  <- the two tests are algebraically the same")

# CI for the difference in proportions (unpooled SE)
se_diff = np.sqrt(p_a*(1-p_a)/n_a + p_b*(1-p_b)/n_b)
zc = stats.norm.ppf(0.975)
print(f"\n95% CI for the lift: [{(p_b-p_a) - zc*se_diff:+.4f}, {(p_b-p_a) + zc*se_diff:+.4f}]")
print(f"Cramer's V = {cramers_v(ab):.4f} (tiny -- typical for conversion tests)")

### (b) Categorical feature selection

The chi-square statistic ranks categorical features by how strongly they relate to a
categorical target. `sklearn.feature_selection.chi2` does this at scale — but remember the
multiple-comparisons warning from Notebook 6: screening 500 features at $\alpha = 0.05$ will
hand you 25 false positives by construction.

In [ ]:
# Which categorical features relate to churn?
m = 3_000
data = pd.DataFrame({
    "contract":     rng.choice(["monthly", "yearly", "two_year"], m, p=[0.55, 0.30, 0.15]),
    "payment":      rng.choice(["card", "bank", "cheque"], m, p=[0.5, 0.3, 0.2]),
    "support_tier": rng.choice(["basic", "plus", "premium"], m, p=[0.6, 0.3, 0.1]),
    "region":       rng.choice(["N", "S", "E", "W"], m),
})
# churn depends on contract and support_tier only; payment and region are noise
risk = (0.34 * (data.contract == "monthly") + 0.10 * (data.contract == "yearly")
        + 0.12 * (data.support_tier == "basic") + 0.04)
data["churn"] = (rng.random(m) < risk).astype(int)

rows = []
for col in ["contract", "payment", "support_tier", "region"]:
    tbl = pd.crosstab(data[col], data["churn"])
    c2, pv, dof, _ = stats.chi2_contingency(tbl)
    rows.append({"feature": col, "chi2": round(c2, 2), "df": dof,
                 "p_value": f"{pv:.2e}", "cramers_v": round(cramers_v(tbl), 4)})

ranking = pd.DataFrame(rows).sort_values("chi2", ascending=False)
print(ranking.to_string(index=False))
print("\nThe two genuine drivers rise to the top; the two noise features do not.")
print(f"\nChurn rate by contract:\n{data.groupby('contract')['churn'].mean().round(3).to_string()}")

In [ ]:
# The same ranking with scikit-learn, for pipeline use
from sklearn.feature_selection import chi2 as sk_chi2
from sklearn.preprocessing import OneHotEncoder

enc = OneHotEncoder(sparse_output=False)
X_enc = enc.fit_transform(data[["contract", "payment", "support_tier", "region"]])
scores, pvals = sk_chi2(X_enc, data["churn"])

sk = pd.DataFrame({"category": enc.get_feature_names_out(), "chi2": scores.round(2),
                   "p_value": [f"{p:.2e}" for p in pvals]}).sort_values("chi2", ascending=False)
print(sk.head(8).to_string(index=False))
print("\nNote this scores individual CATEGORIES, not whole features -- useful for")
print("spotting which level of a feature carries the signal.")

### (c) Checking whether a classifier is calibrated

A **calibrated** model's predicted probabilities match reality: among cases it says are 70%
likely, about 70% should be positive. Bin the predictions and run a goodness-of-fit test on
observed vs expected positives — the **Hosmer–Lemeshow** test.

In [ ]:
# Simulate a well-calibrated model and an over-confident one
truth_p = rng.random(4_000)
y_true = (rng.random(4_000) < truth_p).astype(int)

pred_good = truth_p                                       # perfectly calibrated
pred_bad  = np.clip(truth_p * 1.6 - 0.15, 0.001, 0.999)   # systematically over-confident

def hosmer_lemeshow(y, p, bins=10):
    '''Hosmer-Lemeshow goodness-of-fit test for probability calibration.'''
    frame = pd.DataFrame({"y": y, "p": p})
    frame["bin"] = pd.qcut(frame["p"], bins, labels=False, duplicates="drop")
    g = frame.groupby("bin").agg(n=("y", "size"), obs=("y", "sum"), exp=("p", "sum"))
    stat = (((g.obs - g.exp) ** 2) / (g.exp * (1 - g.exp / g.n))).sum()
    dof = len(g) - 2
    return stat, stats.chi2(dof).sf(stat), g

for name, pred in [("calibrated model", pred_good), ("over-confident model", pred_bad)]:
    stat, pv, g = hosmer_lemeshow(y_true, pred)
    verdict = "calibration OK" if pv > 0.05 else "MISCALIBRATED"
    print(f"{name:<22} HL chi2 = {stat:7.3f}  p = {pv:.4f}  -> {verdict}")

_, _, g_bad = hosmer_lemeshow(y_true, pred_bad)
_, _, g_good = hosmer_lemeshow(y_true, pred_good)

plt.plot([0, 1], [0, 1], "k--", label="perfect calibration")
plt.plot(g_good.exp/g_good.n, g_good.obs/g_good.n, "o-", color="seagreen", label="calibrated model")
plt.plot(g_bad.exp/g_bad.n,  g_bad.obs/g_bad.n,  "o-", color="crimson", label="over-confident model")
plt.xlabel("mean predicted probability"); plt.ylabel("observed frequency")
plt.title("Calibration plot")
plt.legend(fontsize=8)
plt.show()

---
## 8.8 Choosing among the categorical tests

```
Both variables categorical?

ONE variable, compared to expected proportions ....... chi-square goodness-of-fit
TWO variables, one sample .............................. chi-square independence
ONE variable, several independent samples .............. chi-square homogeneity
2x2 table with any expected count < 5 .................. Fisher's exact test
PAIRED / repeated measures on the same subjects ........ McNemar's test
More than 2 paired conditions .......................... Cochran's Q test
Ordered categories (want to detect a trend) ............ Cochran-Armitage trend test
Very sparse large table ................................ permutation chi-square
```

**Reporting template**

> $\chi^2(df) = \text{statistic}, p = \text{p-value}, V = \text{Cramér's V}, n = \text{total}$
>
> *"Subscription rate differed by device type, $\chi^2(2) = 41.6$, $p < 0.001$, Cramér's
> $V = 0.16$ (weak). Desktop users subscribed at 42% versus 29% on mobile and 26% on
> tablet."*

---
## Exercises

**Exercise 1.** A company claims its support tickets are evenly split across four product
lines. Last month's counts were A: 142, B: 178, C: 96, D: 124.
(a) State the hypotheses and run a goodness-of-fit test.
(b) Which product line deviates most from expectation?
(c) Would the conclusion change if all counts were three times larger with the same
proportions?

In [ ]:
# --- Solution 1 -------------------------------------------------------------
counts = np.array([142, 178, 96, 124])
labels = list("ABCD")
exp = np.full(4, counts.sum() / 4)

res = stats.chisquare(counts)
print("(a) H0: all four product lines generate equal ticket volume")
print("    H1: at least one differs")
print(f"    chi2 = {res.statistic:.4f}, df = 3, p = {res.pvalue:.5f}")
print(f"    Decision: {'REJECT H0' if res.pvalue < 0.05 else 'fail to reject H0'}")

contrib = (counts - exp) ** 2 / exp
resid = (counts - exp) / np.sqrt(exp)
print("\n(b)", pd.DataFrame({"line": labels, "observed": counts, "expected": exp,
                             "std_residual": resid.round(2),
                             "contribution": contrib.round(2)}).to_string(index=False))
print(f"    Largest contributor: line {labels[int(np.argmax(contrib))]} "
      f"({'above' if resid[np.argmax(contrib)] > 0 else 'below'} expectation)")

tripled = counts * 3
res3 = stats.chisquare(tripled)
print(f"\n(c) With counts tripled: chi2 = {res3.statistic:.4f}, p = {res3.pvalue:.3e}")
print(f"    chi2 scales linearly with n ({res.statistic:.2f} * 3 = {res.statistic*3:.2f}),")
print("    so the same proportions become far more significant. Proportions unchanged,")
print("    evidence stronger -- exactly the significance-vs-effect-size distinction.")

**Exercise 2.** A hospital records treatment outcome by treatment type. Test for
independence, report the effect size, and identify which cells drive the association.

In [ ]:
# --- Solution 2 -------------------------------------------------------------
hosp = pd.DataFrame([[ 78, 42, 20],
                     [ 95, 30, 15],
                     [ 52, 48, 40]],
                    index=["drug A", "drug B", "placebo"],
                    columns=["improved", "no_change", "worsened"])

c2, pv, dof, ex = stats.chi2_contingency(hosp)
print(hosp)
print(f"\nchi2({dof}) = {c2:.4f}, p = {pv:.3e}, Cramer's V = {cramers_v(hosp):.4f}")
print(f"Minimum expected count = {ex.min():.1f} (assumption satisfied)")

O_, N_ = hosp.to_numpy(), hosp.to_numpy().sum()
R_, C_ = O_.sum(axis=1)[:, None], O_.sum(axis=0)[None, :]
adj = (O_ - ex) / np.sqrt(ex * (1 - R_/N_) * (1 - C_/N_))
print("\nAdjusted residuals:")
print(pd.DataFrame(adj.round(2), index=hosp.index, columns=hosp.columns))

print("\nCells with |r| > 2:")
for i, r_ in enumerate(hosp.index):
    for j, c_ in enumerate(hosp.columns):
        if abs(adj[i, j]) > 2:
            print(f"  {r_:<8} / {c_:<10} {adj[i,j]:+.2f}  "
                  f"({'more' if adj[i,j] > 0 else 'fewer'} than expected)")
print("\nImprovement rate by treatment:")
print((hosp['improved'] / hosp.sum(axis=1)).round(3).to_string())

**Exercise 3.** A pilot study of a rare side effect gives the 2×2 table below.
(a) Why is a plain chi-square test inappropriate?
(b) Run Fisher's exact test and interpret the odds ratio.
(c) Compare all three approaches (uncorrected, Yates, Fisher).
(d) How many patients per arm would you need for the chi-square approximation to be safe?

In [ ]:
# --- Solution 3 -------------------------------------------------------------
pilot = np.array([[2, 28],       # treatment: side effect, none
                  [8, 22]])      # control

print(pd.DataFrame(pilot, index=["treatment", "control"], columns=["side_effect", "none"]))
_, _, _, ex_p = stats.chi2_contingency(pilot)
print(f"\nExpected counts:\n{np.round(ex_p, 2)}")
print(f"(a) Minimum expected count = {ex_p.min():.2f} < 5, so the Normal approximation")
print("    behind chi-square is not reliable here.\n")

odds, p_f = stats.fisher_exact(pilot)
print(f"(b) Fisher's exact: odds ratio = {odds:.4f}, p = {p_f:.4f}")
print(f"    OR = {odds:.3f} means the odds of a side effect in the treatment group are")
print(f"    about {odds:.2f}x those in the control group -- i.e. {1/odds:.1f}x LOWER.")

_, p_u, _, _ = stats.chi2_contingency(pilot, correction=False)
_, p_y, _, _ = stats.chi2_contingency(pilot, correction=True)
print(f"\n(c) uncorrected chi-square p = {p_u:.4f}")
print(f"    Yates-corrected      p = {p_y:.4f}")
print(f"    Fisher's exact       p = {p_f:.4f}   <- report this one")

# (d) find the scale at which min expected count reaches 5
props = pilot / pilot.sum()
for total in range(40, 601, 10):
    tbl = (props * total)
    r_, c_ = tbl.sum(axis=1)[:, None], tbl.sum(axis=0)[None, :]
    min_e = (r_ * c_ / total).min()
    if min_e >= 5:
        print(f"\n(d) Expected counts reach 5 at about n = {total} total "
              f"({total//2} per arm), min E = {min_e:.1f}")
        break

**Exercise 4 (challenge).** You run a chi-square test on a 5×4 contingency table with
$n = 80$ and get $p = 0.03$.
(a) List everything that could make this result untrustworthy.
(b) Simulate the situation to show how often a 5×4 table with $n = 80$ produces
$p < 0.05$ when the variables are genuinely independent — using the chi-square
approximation, and using a permutation test.
(c) What would you do instead?

In [ ]:
# --- Solution 4 -------------------------------------------------------------
print("(a) Concerns with a 5x4 table at n = 80:")
print("    * 20 cells and only 80 observations -> average 4 per cell, so many expected")
print("      counts fall below 5 and the chi-square approximation is unreliable")
print("    * df = (5-1)(4-1) = 12, a lot of freedom to find noise")
print("    * p = 0.03 is marginal; it would not survive any multiplicity correction")
print("    * a significant result says nothing about strength -- check Cramer's V")
print("    * if this table was one of several you looked at, the real error rate is higher\n")

# (b) Simulate under true independence
r_dim, c_dim, n_obs, sims = 5, 4, 80, 3_000
row_p = np.array([0.30, 0.25, 0.20, 0.15, 0.10])
col_p = np.array([0.40, 0.30, 0.20, 0.10])

rej_approx = 0
vs = []
for _ in range(sims):
    ri = rng.choice(r_dim, n_obs, p=row_p)
    ci = rng.choice(c_dim, n_obs, p=col_p)          # independent by construction
    tbl = np.zeros((r_dim, c_dim), dtype=int)
    np.add.at(tbl, (ri, ci), 1)
    if (tbl.sum(axis=1) == 0).any() or (tbl.sum(axis=0) == 0).any():
        continue
    c2_, pv_, _, _ = stats.chi2_contingency(tbl, correction=False)
    rej_approx += pv_ < 0.05
    vs.append(np.sqrt(c2_ / (n_obs * (min(tbl.shape) - 1))))

print(f"(b) True independence, {sims:,} simulated 5x4 tables with n = 80:")
print(f"    chi-square approximation rejects at 5% in {rej_approx/sims:.4f} of tables")
print(f"    (nominal rate is 0.0500 -- the approximation is off in sparse tables)")
print(f"    Median Cramer's V under pure independence: {np.median(vs):.3f}")
print(f"    95th percentile of V under independence  : {np.quantile(vs, 0.95):.3f}")
print("    So any V below about 0.3 in a table this sparse is unremarkable.\n")

print("(c) What to do instead:")
print("    1. Collapse categories so every expected count is >= 5")
print("    2. Use a permutation or Monte-Carlo chi-square test rather than the")
print("       asymptotic p-value")
print("    3. Report Cramer's V with a confidence interval, not just p")
print("    4. Treat p = 0.03 on 12 df with n = 80 as a hypothesis to test on new data,")
print("       not a finding")

---
## Summary

| Test | Question | scipy | df |
|---|---|---|---|
| Goodness-of-fit | Does one variable match expected proportions? | `chisquare(obs, exp)` | $k-1$ |
| Independence | Are two variables associated? | `chi2_contingency(table)` | $(r-1)(c-1)$ |
| Homogeneity | Same distribution across populations? | `chi2_contingency(table)` | $(r-1)(c-1)$ |
| Fisher's exact | 2×2 with small counts | `fisher_exact(table)` | — |
| McNemar | Paired binary data | `binomtest(b, b+c)` | 1 |

**Key formulas**

$$\chi^2 = \sum \frac{(O-E)^2}{E}, \qquad
E_{ij} = \frac{R_i C_j}{N}, \qquad
V = \sqrt{\frac{\chi^2}{n \min(r-1, c-1)}}$$

**Rules to carry away**

1. Chi-square works on **counts**, never on percentages
2. Check that expected counts are adequate; use **Fisher** when they are not
3. Paired data needs **McNemar**, not chi-square
4. Always report **Cramér's V** alongside $p$
5. Use **adjusted residuals** to say *where* the association lives
6. A significant $\chi^2$ shows association, never causation

---

## 🎓 Course complete

You now have the full toolkit:

| Notebook | What it gave you |
|---|---|
| [1. Probability Basics](1.%20Probability%20Basics.ipynb) | The language of uncertainty; Bayes' theorem |
| [2. Random Variables](2.%20Random%20Variables.ipynb) | Expectation, variance, distributions of numbers |
| [3. Probability Distributions](3.%20Probability%20Distributions.ipynb) | The named templates for randomness |
| [4. Sampling Techniques](4.%20Sampling%20Techniques.ipynb) | How to get data you can generalise from; the CLT |
| [5. Correlation and Covariance](5.%20Correlation%20and%20Covariance.ipynb) | Relationships between variables, and their traps |
| [6. Hypothesis Testing](6.%20Hypothesis%20Testing.ipynb) | The framework for "is this real?" |
| [7. t-Test](7.%20t-Test.ipynb) | Comparing means, properly |
| [8. Chi-Square Tests](8.%20Chi-Square%20Test.ipynb) | Comparing counts and categories |

**The five habits that matter more than any formula**

1. **Plot the data first.** Anscombe's quartet, every time.
2. **Report effect sizes and confidence intervals**, not just p-values.
3. **Check your assumptions**, and know the fallback when they fail.
4. **Ask where the data came from** before you trust any number computed from it.
5. **Correlation is not causation** — say it out loud before every conclusion.

Where to go next: linear and logistic regression, ANOVA, Bayesian inference, and causal
inference all build directly on what is in these eight notebooks.